# GCOL-OBS-PATCH.10H.0R.1 — Contrôle X22 SPW25 (en-tête FITS + SHA-256)

**Nature** : pré-audit documentaire, aveugle aux amplitudes des raies. Ce notebook s'exécute **dans Google Colab connecté au Google Drive de l'auteur** ; il n'a pas été exécuté sur le fichier de 1,35 Go depuis l'assistant, en raison de la limite de transfert de 256 Mo.

**Architecture à quatre niveaux :**

1. **CHH/GVH (hypothèse)** : aucune revendication physique ; `GVH_USED=False`, aucune dérivée intrinsèque CHH.
2. **Physique et provenance établies** : métadonnées FITS/ALMA, conventions spectrales, préenregistrement figé.
3. **Audit** : nom, taille, en-tête, couverture de fréquence et SHA-256 du fichier entier **sans accès aux pixels ni aux intensités**.
4. **Verdict** : état documentaire uniquement, sans démasquage des raies.

**Important** : aucun changement des fenêtres de vitesse, aucun ajustement des seuils, aucune validation de la couche au premier plan. Le correctif SPW25 proposé demeure **NOT_ADOPTED** tant que la décision scientifique n'est pas documentée.

## 1 — Connexion à Drive et verrou de non-inspection

Exécuter dans Google Colab. Drive est monté en lecture dans ce notebook : aucune sauvegarde n'est effectuée dans le dossier de projet. Le fichier de sortie JSON sera créé dans `/content/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, json, hashlib, math, sys, subprocess
try:
    from astropy.io import fits
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'astropy'])
    from astropy.io import fits

ROOT = Path('/content/drive/MyDrive/GCOL_OBS_0_3_2_1_1_PATCH10H0')
X22_DIR = ROOT / 'ALMA_2015.1.00147.S' / 'X22'
NAME = 'member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw25.cube.I.pbcor.fits'
PATH = X22_DIR / NAME
EXPECTED_BYTES = 1349115840
FROZEN_WINDOWS_KMS = {'core': [6.40, 7.00], 'foreground_test': [7.70, 8.30]}
PRE_REGISTERED_REST_GHZ = 363.94589  # transition cible, correspondance SPW25 proposée non adoptée
C_KMS = 299792.458
GVH_USED = False
INSPECT_LINE_AMPLITUDES = False
ALLOW_RETUNING = False
assert not (GVH_USED or INSPECT_LINE_AMPLITUDES or ALLOW_RETUNING)
assert PATH.is_file(), f'Fichier non trouvé: {PATH}'
assert PATH.stat().st_size == EXPECTED_BYTES, 'Taille différente de la valeur Drive inventoriée'
print('Fichier présent, nom et taille confirmés:', PATH.name, EXPECTED_BYTES, 'octets')

Mounted at /content/drive
Fichier présent, nom et taille confirmés: member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw25.cube.I.pbcor.fits 1349115840 octets


## 2 — Audit FITS en-tête seulement

La fonction `fits.getheader` ne lit pas le tableau des intensités. On recherche les axes, unités, la fréquence de repos, le repère spectral, les paramètres WCS et le nombre de canaux. Les références aux fenêtres de vitesse ne sont utilisées **que pour tester la couverture de l'axe**.

In [ ]:
h = fits.getheader(PATH, 0)  # header only — NEVER use fits.getdata()
axes = []
for i in range(1, int(h.get('NAXIS', 0)) + 1):
    axes.append({'axis': i, 'ctype': str(h.get(f'CTYPE{i}', '')),
                 'cunit': str(h.get(f'CUNIT{i}', '')),
                 'naxis': int(h.get(f'NAXIS{i}', 0)),
                 'crval': float(h.get(f'CRVAL{i}', 0)),
                 'cdelt': float(h.get(f'CDELT{i}', 0)),
                 'crpix': float(h.get(f'CRPIX{i}', 1))})
spectral = [a for a in axes if 'FREQ' in a['ctype'].upper()]
assert len(spectral) == 1, f'Attendu un seul axe fréquentiel : {spectral}'
a = spectral[0]
assert a['naxis'] >= 2 and a['cdelt'] != 0
unit = a['cunit'].strip().lower()
unitfactor = {'hz': 1.0, 'khz': 1e3, 'mhz': 1e6, 'ghz': 1e9}.get(unit)
assert unitfactor is not None, f'Unité fréquentielle inconnue : {unit}'
# FITS/astropy frequency values from pixel indices 0 and n-1, no data plane.
f0 = (a['crval'] + (1-a['crpix'])*a['cdelt'])*unitfactor
f1 = (a['crval'] + (a['naxis']-a['crpix'])*a['cdelt'])*unitfactor
f_low, f_high = sorted((f0, f1))
rest = h.get('RESTFRQ', h.get('RESTFREQ'))
rest = float(rest) if rest is not None else None
specsys = str(h.get('SPECSYS', 'UNKNOWN'))
nominal = PRE_REGISTERED_REST_GHZ * 1e9
nu_windows = {label: sorted([nominal*(1-v/C_KMS) for v in vr]) for label, vr in FROZEN_WINDOWS_KMS.items()}
window_coverage = {label: (f_low <= lo and hi <= f_high) for label, (lo,hi) in nu_windows.items()}
header_audit = {
    'header_only': True,
    'naxis': int(h.get('NAXIS', 0)),
    'axes': axes,
    'object': str(h.get('OBJECT','')),
    'bunit': str(h.get('BUNIT','')),
    'specsys': specsys,
    'restfreq_hz': rest,
    'restfreq_compatible_with_prereg_target_ghz': (abs(rest/1e9-PRE_REGISTERED_REST_GHZ)<0.0001 if rest else False),
    'channel_count': a['naxis'],
    'frequency_low_hz': f_low,
    'frequency_high_hz': f_high,
    'nominal_radio_velocity_windows_coverage': window_coverage,
    'velocity_convention_used_for_metadata_check': 'radio; nominal-rest-frequency; no line data inspected',
}
print(json.dumps(header_audit, indent=2, ensure_ascii=False))
assert all(window_coverage.values()), 'Une fenêtre gelée est hors couverture de fréquence'


{
  "header_only": true,
  "naxis": 4,
  "axes": [
    {
      "axis": 1,
      "ctype": "RA---SIN",
      "cunit": "deg",
      "naxis": 840,
      "crval": 52.29375,
      "cdelt": -9.166666666667e-06,
      "crpix": 421.0
    },
    {
      "axis": 2,
      "ctype": "DEC--SIN",
      "cunit": "deg",
      "naxis": 840,
      "crval": 31.22541666667,
      "cdelt": 9.166666666667e-06,
      "crpix": 421.0
    },
    {
      "axis": 3,
      "ctype": "FREQ",
      "cunit": "Hz",
      "naxis": 478,
      "crval": 363908317596.0,
      "cdelt": 122189.173584,
      "crpix": 1.0
    },
    {
      "axis": 4,
      "ctype": "STOKES",
      "cunit": "",
      "naxis": 1,
      "crval": 1.0,
      "cdelt": 1.0,
      "crpix": 1.0
    }
  ],
  "object": "NGC1333_IRAS4A",
  "bunit": "Jy/beam",
  "specsys": "LSRK",
  "restfreq_hz": 363945894000.0,
  "restfreq_compatible_with_prereg_target_ghz": true,
  "channel_count": 478,
  "frequency_low_hz": 363908317596.0,
  "frequency_high_hz": 36396660

## 3 — Empreinte SHA-256 (facultative mais recommandée)

Pour un cube de 1,35 Go, ce calcul peut consommer des lectures Drive. Il lit uniquement les octets pour le hachage, **sans les décoder en intensités ni produire de cartes/spectres**. Conserver le SHA avec le nom exact du fichier. Aucun SHA indépendant ALMA n'est présumé.

In [ ]:
RUN_SHA256 = True
sha256 = None
if RUN_SHA256:
    hs = hashlib.sha256()
    with open(PATH, 'rb') as fh:
        for chunk in iter(lambda: fh.read(8 * 1024 * 1024), b''):
            hs.update(chunk)
    sha256 = hs.hexdigest()
    print('SHA-256 mesuré :', sha256)
else:
    print('SHA-256 non calculé (contrôle incomplet).')

SHA-256 mesuré : 67fd0dee3db8f9ba97db1f63d0eb655582065d1f6f57f16e1d046a3eae728a78


## 4 — Certificat provisoire (sans démasquage)

Un résultat positif certifie seulement l'en-tête, l'axe et l'intégrité calculée. **Ne jamais interpréter PASS comme validation du test d'absorption**.

In [ ]:
report = {
  'patch': 'GCOL-OBS-PATCH.10H.0R.1',
  'file': NAME, 'file_size_bytes': PATH.stat().st_size,
  'sha256': sha256,
  'header': header_audit,
  'frozen_velocity_windows_kms': FROZEN_WINDOWS_KMS,
  'original_contract_modified': False,
  'proposed_spw25_amendment_adopted': False,
  'line_amplitudes_inspected': False,
  'thresholds_retuned': False,
  'GVH_USED': False,
  'IRAS4_FOREGROUND_LAYER_CERTIFIED': False,
  'FRONT_BACK_LAYER_ORDER_DERIVED': False,
  'LINE_TEST_AUTHORIZED': False,
  'decision': 'X22_SPW25_METADATA_READY_ONLY__ABSTAIN_LINE_TEST'
}
out = Path('/content/PATCH10H0R1_X22_SPW25_header_sha256_blind_report.json')
out.write_text(json.dumps(report, indent=2, ensure_ascii=False))
assert report['line_amplitudes_inspected'] is False
assert report['LINE_TEST_AUTHORIZED'] is False
print('Certificat local Colab :', out)
print('DÉCISION :', report['decision'])

Certificat local Colab : /content/PATCH10H0R1_X22_SPW25_header_sha256_blind_report.json
DÉCISION : X22_SPW25_METADATA_READY_ONLY__ABSTAIN_LINE_TEST


### Étape scientifique suivante

Après lecture du rapport, vérifier la provenance des produits et les correspondances `SPW ↔ RESTFRQ` des deux membres. L'amendement des SPW reste proposé, **non adopté**. Ne pas activer le test d'absorption avant décision explicite sur le préenregistrement et les contrôles négatifs.